<h1 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  🏥 Pre-Auth Pipeline — Agent 1: Document Intelligence
</h1>

**Pipeline Overview:**
```
User Uploads Medical Documents
        |
        v
>>> [AGENT 1] Document Intelligence Agent  <<<
    (OCR + Medical Entity Extraction)
    - Lab Report (PNG)
    - Doctor Notes / Procedure Template (PNG)
    - Patient Information Sheet (PNG)
    - Insurance Card (PNG)
    - Pre-Treatment Estimate (PDF -> images)
        |
        v
    Structured JSON Output
```

**Model Used:** `amazon.nova-pro-v1:0` — best multimodal reasoning for complex clinical documents.

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4); color: white; padding: 15px; border-radius: 10px; font-family: Arial, sans-serif;">
  ⚙️ Setup & Dependencies
</h2>

In [ ]:
# Install dependencies
# pdf2image converts PDF pages to images so Nova Pro can read them
# !pip install boto3 pdf2image Pillow pymupdf --quiet

In [ ]:
import boto3
import json
import base64
import fitz          # PyMuPDF — converts PDF pages to PNG bytes
from pathlib import Path
from PIL import Image
import io

print("✅ All imports successful")

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4); color: white; padding: 15px; border-radius: 10px; font-family: Arial, sans-serif;">
  🔧 AWS Boto3 Client Setup
</h2>

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AWS BOTO3 SETUP
# The hackathon sandbox credentials are picked up automatically from:
#   1. Environment variables  (AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
#   2. IAM Role attached to the SageMaker / EC2 instance
#   3. ~/.aws/credentials  (if running locally)
# No explicit key passing needed — boto3 uses the credential chain.
# ─────────────────────────────────────────────────────────────────────────────

AWS_REGION = "us-east-1"   # ← change if your sandbox is in a different region

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
)

# ─── Model IDs ────────────────────────────────────────────────────────────────
PRO_MODEL_ID   = "amazon.nova-pro-v1:0"    # best vision + reasoning → Agent 1
LITE_MODEL_ID  = "amazon.nova-lite-v1:0"   # fast multimodal  → later agents
MICRO_MODEL_ID = "amazon.nova-micro-v1:0"  # text-only / cheapest → policy checks

print(f"✅ Bedrock client created in region: {AWS_REGION}")
print(f"   Using model: {PRO_MODEL_ID}")

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4); color: white; padding: 15px; border-radius: 10px; font-family: Arial, sans-serif;">
  📂 Document Loading Utilities
</h2>

We have **5 source documents** for this patient:

| File | Type | Contains |
|------|------|----------|
| `lab_report_ai.png` | PNG image | Lab results, patient ID, doctor, insurer ref |
| `docotr_notes_ai.png` | PNG image | Procedure details, ICD-10, CPT, clinical findings |
| `patient_ai.png` | PNG image | Demographics, insurance policy number, pharmacy |
| `insurance_card_ai.png` | PNG image | BCBS card — policy/group numbers, co-pays |
| `medical_pretreatment_estimate_ai.pdf` | PDF | CPT codes, cost estimates, treatment description |

In [ ]:
def image_to_base64(image_path: str, max_size: tuple = (1600, 1600)) -> dict:
    """
    Load an image file, optionally resize it, and return a Nova-compatible
    image content block dict.

    Nova Pro supports: jpeg, png, gif, webp
    Payload limit: 25 MB total across all images.
    """
    path = Path(image_path)
    fmt = path.suffix.lstrip(".").lower()
    if fmt == "jpg":
        fmt = "jpeg"

    img = Image.open(path)
    img.thumbnail(max_size, Image.LANCZOS)   # resize in-place, preserves aspect ratio

    buffer = io.BytesIO()
    img.save(buffer, format=fmt.upper())
    b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    print(f"  📷 {path.name}  [{img.size[0]}×{img.size[1]}px, {len(b64)//1024} KB b64]")
    return {
        "image": {
            "format": fmt,
            "source": {"bytes": b64}
        }
    }


def pdf_pages_to_base64(pdf_path: str, dpi: int = 150) -> list:
    """
    Convert each page of a PDF to a PNG image and return a list of
    Nova-compatible image content block dicts.

    Uses PyMuPDF (fitz) — no Poppler dependency required.
    """
    doc = fitz.open(pdf_path)
    blocks = []
    for page_num, page in enumerate(doc):
        zoom = dpi / 72          # 72 DPI is PDF default
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)
        png_bytes = pix.tobytes("png")
        b64 = base64.b64encode(png_bytes).decode("utf-8")
        print(f"  📄 {Path(pdf_path).name} — page {page_num+1}  [{pix.width}×{pix.height}px, {len(b64)//1024} KB b64]")
        blocks.append({
            "image": {
                "format": "png",
                "source": {"bytes": b64}
            }
        })
    return blocks


print("✅ Document utility functions defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DOCUMENT PATHS  — update these if running from a different working directory
# ─────────────────────────────────────────────────────────────────────────────
DOCS = {
    "lab_report":            "lab_report_ai.png",
    "doctor_notes":          "docotr_notes_ai.png",
    "patient_info":          "patient_ai.png",
    "insurance_card":        "insurance_card_ai.png",
    "pretreatment_estimate": "medical_pretreatment_estimate_ai.pdf",
}

print("Loading documents...")
image_blocks = []

# Load PNG images
for name in ["lab_report", "doctor_notes", "patient_info", "insurance_card"]:
    image_blocks.append(image_to_base64(DOCS[name]))

# Load PDF (converts each page to PNG)
image_blocks.extend(pdf_pages_to_base64(DOCS["pretreatment_estimate"]))

print(f"\n✅ Total image blocks ready for Nova Pro: {len(image_blocks)}")

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4); color: white; padding: 15px; border-radius: 10px; font-family: Arial, sans-serif;">
  🤖 Agent 1 — Document Intelligence (OCR + Entity Extraction)
</h2>

Sends **all 5 documents** to Nova Pro in a single multimodal call.  
The model is instructed to extract a structured JSON object with every field  
needed downstream by the authorization pipeline.

In [ ]:
# ─── System Prompt ────────────────────────────────────────────────────────────
AGENT1_SYSTEM = [
    {
        "text": """You are a highly accurate medical document intelligence agent.
Your job is to read a set of clinical documents (lab reports, doctor notes,
patient information sheets, insurance cards, and cost estimates) and extract
all relevant fields required for insurance prior-authorization.

RULES:
- Return ONLY a valid JSON object. No markdown, no preamble, no explanation.
- If a field is not found in any document, set its value to null.
- Normalize all field names to snake_case.
- For dates, use YYYY-MM-DD format.
- For currency amounts, return numeric values only (no $ sign).
- Extract ALL CPT codes and ALL ICD-10 codes you can find (as arrays).
"""
    }
]

print("✅ System prompt defined")

In [ ]:
# ─── Extraction Schema Prompt ─────────────────────────────────────────────────
EXTRACTION_PROMPT = """You are given the following medical documents as images.
Read every document carefully and extract ALL of the fields listed below.

Return a single JSON object matching EXACTLY this schema:

{
  "patient": {
    "name": string,
    "dob": string,              // YYYY-MM-DD
    "age": integer,
    "gender": string,
    "mrn": string,              // Medical Record Number / Patient ID
    "address": string,
    "city": string,
    "state": string,
    "zip": string,
    "phone": string,
    "email": string,
    "emergency_contact_name": string,
    "emergency_contact_phone": string,
    "emergency_contact_relationship": string
  },
  "insurance": {
    "insurer_name": string,
    "plan_type": string,        // PPO, HMO, EPO, etc.
    "policy_number": string,
    "group_number": string,
    "member_id": string,
    "policyholder_name": string,
    "relationship_to_patient": string,
    "copay_office_visit": number,
    "copay_er": number,
    "copay_specialist": number,
    "administered_by": string,
    "underwritten_by": string,
    "bin": string,
    "pcn": string,
    "rxgrp": string,
    "customer_service_phone": string
  },
  "clinical": {
    "diagnosis": string,                   // human-readable
    "icd10_codes": [string],               // e.g. ["M54.16"]
    "chief_complaint": string,
    "symptom_duration_weeks": integer,
    "pain_score": integer,
    "clinical_findings": string,
    "prior_treatments": [string],          // list of previously tried treatments
    "requested_procedure": string,
    "cpt_codes": [string],                 // e.g. ["72148", "72148-26", "99204"]
    "ordering_physician": string,
    "physician_phone": string,
    "physician_specialty": string
  },
  "facility": {
    "hospital_name": string,
    "hospital_address": string,
    "date_of_service": string,             // YYYY-MM-DD
    "date_of_proposed_treatment": string   // YYYY-MM-DD
  },
  "cost_estimate": {
    "line_items": [
      {
        "description": string,
        "cpt_code": string,
        "estimated_price": number
      }
    ],
    "total_estimated_cost": number
  },
  "lab_results": {
    "hba1c": string,
    "cholesterol_total": string,
    "ldl": string,
    "hdl": string,
    "triglycerides": string,
    "creatinine": string,
    "egfr": string,
    "alt": string,
    "ast": string
  },
  "pharmacy": {
    "pharmacy_name": string,
    "pharmacy_address": string,
    "pharmacy_phone": string
  },
  "prescribed_medications": [string],     // list of medications prescribed
  "extraction_confidence": "high" | "medium" | "low",
  "missing_fields": [string]              // list any fields you could not find
}
"""

print("✅ Extraction schema prompt defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT 1 — NOVA PRO MULTIMODAL API CALL
# All 5 document images are sent in a single request alongside the schema prompt.
# Nova Pro supports up to 25 MB payload and multiple images per turn.
# ─────────────────────────────────────────────────────────────────────────────

# Build the user message content: all image blocks FIRST, then the text prompt
user_content = image_blocks + [{"text": EXTRACTION_PROMPT}]

message_list = [
    {
        "role": "user",
        "content": user_content
    }
]

inf_params = {
    "max_new_tokens": 2000,
    "temperature": 0.1,    # low temperature → deterministic extraction
    "top_p": 0.9,
    "top_k": 20
}

native_request = {
    "messages": message_list,
    "system": AGENT1_SYSTEM,
    "inferenceConfig": inf_params,
}

print("🚀 Sending all documents to Amazon Nova Pro...")
print(f"   Total content blocks: {len(user_content)} ({len(image_blocks)} images + 1 prompt)")

response = bedrock_client.invoke_model(
    modelId=PRO_MODEL_ID,
    body=json.dumps(native_request)
)

request_id = response["ResponseMetadata"]["RequestId"]
model_response = json.loads(response["body"].read())

print(f"\n✅ Response received  |  RequestId: {request_id}")
print(f"   Stop reason: {model_response.get('stopReason', 'N/A')}")
print(f"   Input tokens:  {model_response.get('usage', {}).get('inputTokens', 'N/A')}")
print(f"   Output tokens: {model_response.get('usage', {}).get('outputTokens', 'N/A')}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARSE & VALIDATE THE EXTRACTED JSON
# ─────────────────────────────────────────────────────────────────────────────

raw_text = model_response["output"]["message"]["content"][0]["text"]

# Strip accidental markdown fences (```json ... ```) if model adds them
clean_text = raw_text.strip()
if clean_text.startswith("```"):
    clean_text = clean_text.split("```")[1]
    if clean_text.lower().startswith("json"):
        clean_text = clean_text[4:]
clean_text = clean_text.strip()

try:
    extracted_data = json.loads(clean_text)
    print("✅ JSON parsed successfully")
except json.JSONDecodeError as e:
    print(f"❌ JSON parse error: {e}")
    print("Raw output:\n", raw_text)
    extracted_data = {}

# Pretty print
print("\n" + "="*70)
print("📋  EXTRACTED MEDICAL ENTITIES")
print("="*70)
print(json.dumps(extracted_data, indent=2))

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4); color: white; padding: 15px; border-radius: 10px; font-family: Arial, sans-serif;">
  ✅ Downstream Handoff — Standardized Output
</h2>

The cell below flattens the extracted data into the **compact downstream format**  
used by Agent 2 (Policy Query Requirement Checker).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT 1 OUTPUT — compact handoff object for the next agent in the pipeline
# ─────────────────────────────────────────────────────────────────────────────

def build_handoff(data: dict) -> dict:
    """Flatten the rich extraction into the canonical downstream format."""
    patient   = data.get("patient", {})
    insurance = data.get("insurance", {})
    clinical  = data.get("clinical", {})
    facility  = data.get("facility", {})
    cost      = data.get("cost_estimate", {})

    return {
        # ── Core fields (required by every downstream agent) ──
        "patient_name":            patient.get("name"),
        "patient_dob":             patient.get("dob"),
        "patient_mrn":             patient.get("mrn"),
        "insurer":                 insurance.get("insurer_name"),
        "plan_type":               insurance.get("plan_type"),
        "policy_number":           insurance.get("policy_number"),
        "group_number":            insurance.get("group_number"),
        "member_id":               insurance.get("member_id"),
        "diagnosis":               clinical.get("diagnosis"),
        "icd10_codes":             clinical.get("icd10_codes", []),
        "icd10":                   (clinical.get("icd10_codes") or [None])[0],  # primary
        "procedure":               clinical.get("requested_procedure"),
        "cpt_codes":               clinical.get("cpt_codes", []),
        "cpt":                     (clinical.get("cpt_codes") or [None])[0],    # primary
        "ordering_physician":      clinical.get("ordering_physician"),
        "physician_specialty":     clinical.get("physician_specialty"),
        "hospital":                facility.get("hospital_name"),
        "date_of_service":         facility.get("date_of_service"),
        "date_of_proposed_treatment": facility.get("date_of_proposed_treatment"),
        "total_estimated_cost":    cost.get("total_estimated_cost"),
        "prior_treatments":        clinical.get("prior_treatments", []),
        "prescribed_medications":  data.get("prescribed_medications", []),
        # ── Metadata ──
        "extraction_confidence":   data.get("extraction_confidence"),
        "missing_fields":          data.get("missing_fields", []),
        "_raw":                    data,   # full extraction for agents that need it
    }


AGENT1_OUTPUT = build_handoff(extracted_data)

print("="*70)
print("🔁  AGENT 1 → AGENT 2 HANDOFF PAYLOAD")
print("="*70)
# Print without the _raw key for readability
display_output = {k: v for k, v in AGENT1_OUTPUT.items() if k != "_raw"}
print(json.dumps(display_output, indent=2))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OPTIONAL: Save the output to disk so subsequent agent notebooks can load it
# ─────────────────────────────────────────────────────────────────────────────

output_path = "agent1_output.json"

with open(output_path, "w") as f:
    json.dump(AGENT1_OUTPUT, f, indent=2)

print(f"💾 Agent 1 output saved to: {output_path}")
print(f"   Next notebook loads it with:  AGENT1_OUTPUT = json.load(open('agent1_output.json'))")
print()
print("─" * 70)
print("PIPELINE STATUS:")
print(f"  ✅ Agent 1 — Document Intelligence    COMPLETE")
print(f"  ⏳ Agent 2 — Policy Query Checker      NEXT")
print(f"  ⏳ Agent 3 — Policy Retrieval (RAG)    PENDING")
print(f"  ⏳ Agent 4 — Document Requirement      PENDING")
print(f"  ⏳ Agent 5 — Eligibility Reasoning     PENDING")
print(f"  ⏳ Agent 6 — Prior Auth Form Gen       PENDING")
print(f"  ⏳ Agent 7 — Portal Automation         PENDING")
print(f"  ⏳ Agent 8 — Claim Status Tracker      PENDING")
print(f"  ⏳ Agent 9 — Appeal Agent              PENDING")
print("─" * 70)

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4); color: white; padding: 15px; border-radius: 10px; font-family: Arial, sans-serif;">
  📝 Implementation Notes
</h2>

### Why Nova Pro for Agent 1?
- Handles **multiple images in one payload** (up to 25 MB)
- Best-in-class OCR accuracy on complex clinical document layouts
- Strong cross-document reasoning (correlates patient IDs across all 5 files)

### Model Selection for the Full Pipeline
| Agent | Task | Recommended Model | Why |
|-------|------|-------------------|-----|
| 1 | Document OCR + extraction | `nova-pro` | Multimodal, high accuracy |
| 2 | Policy field validation | `nova-micro` | Text-only, fast, cheap |
| 3 | RAG policy retrieval | `nova-lite` | Balance of speed & reasoning |
| 4 | Missing doc check | `nova-micro` | Simple rule application |
| 5 | Eligibility reasoning | `nova-pro` | Complex multi-doc reasoning |
| 6 | Form generation | `nova-lite` | Structured text output |
| 7 | Portal automation | `nova-lite` | Tool use / function calling |
| 8 | Status tracking | `nova-micro` | Pattern matching |
| 9 | Appeal drafting | `nova-pro` | Persuasive clinical writing |

### Payload Size Management
Images are resized to max 1600×1600 before encoding.  
For very large document sets, consider processing images in batches  
or uploading to S3 and passing URIs (same approach as the video S3 pattern  
shown in the base notebook).

### Credential Setup for Hackathon Sandbox
```bash
# Option A — Environment variables (recommended for notebooks)
export AWS_ACCESS_KEY_ID=your_key
export AWS_SECRET_ACCESS_KEY=your_secret
export AWS_DEFAULT_REGION=us-east-1

# Option B — AWS CLI profile
aws configure --profile hackathon
# then in the notebook:
# boto3.setup_default_session(profile_name="hackathon")

# Option C — IAM Role (SageMaker / EC2) — no config needed, auto-detected
```